In [7]:
import os
import warnings
import textwrap
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings("ignore")


INPUT_CSV = "/content/cleaned_diabetes_data (1).csv"

OUTPUT_DIR = "/content/eda_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PALETTE = {0: "#4CAF50", 1: "#FF9800", 2: "#F44336"}
LABEL_MAP = {0: "No Diabetes", 1: "Pre-Diabetes", 2: "Diabetes"}
COLORS = [PALETTE[k] for k in sorted(PALETTE)]

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 150, "savefig.bbox": "tight"})

findings = []   # accumulate text findings


def save(fig, name, title=""):
    path = os.path.join(OUTPUT_DIR, name)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print(f"  ✓  {name}")
    return path




print("\n Loading data ")
df = pd.read_csv(INPUT_CSV)
print(f"  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Add readable label column
df["Diabetes_Label"] = df["Diabetes_012"].map(LABEL_MAP)

# Binary (0/1) features
BINARY_COLS = [
    "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "DiffWalk", "Sex"
]

# Ordinal / continuous
ORDINAL_COLS = ["GenHlth", "MentHlth", "PhysHlth", "Age", "Education", "Income"]


# CHART 01 — Target class distribution
# PURPOSE  : Understand class imbalance before any modeling.

print("\n 01  Class distribution ")

counts = df["Diabetes_012"].value_counts().sort_index()
labels = [LABEL_MAP[i] for i in counts.index]
pcts   = counts / counts.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Target Variable Distribution  (Diabetes_012)", fontsize=14, fontweight="bold")

# Bar chart
bars = axes[0].bar(labels, counts.values, color=COLORS, edgecolor="white", linewidth=0.8)
for bar, pct in zip(bars, pcts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
                 f"{pct:.1f}%", ha="center", va="bottom", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Count")
axes[0].set_title("Count per Class")

# Pie chart
axes[1].pie(counts.values, labels=labels, colors=COLORS, autopct="%1.1f%%",
            startangle=140, wedgeprops={"edgecolor": "white", "linewidth": 1.5})
axes[1].set_title("Proportion per Class")

save(fig, "01_class_distribution.png")

findings.append(
    "CLASS IMBALANCE (Chart 01): No Diabetes = 82.7%, Pre-Diabetes = 2.0%, "
    "Diabetes = 15.3%.  The dataset is heavily skewed toward the 'No Diabetes' class. "
    "Modeling phase should apply oversampling (SMOTE) or class-weight adjustments."
)


# CHART 02 — BMI distribution
# PURPOSE  : BMI is a known diabetes risk factor; check its spread and cap.

print(" 02  BMI distribution ")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("BMI Distribution", fontsize=14, fontweight="bold")

# Overall
axes[0].hist(df["BMI"], bins=40, color="#2196F3", edgecolor="white", linewidth=0.5)
axes[0].axvline(df["BMI"].median(), color="red", linestyle="--", label=f'Median={df["BMI"].median():.0f}')
axes[0].axvline(df["BMI"].mean(),   color="orange", linestyle="--", label=f'Mean={df["BMI"].mean():.1f}')
axes[0].set_xlabel("BMI")
axes[0].set_ylabel("Count")
axes[0].set_title("Overall BMI Histogram")
axes[0].legend()

# By class (KDE)
for cls, color in PALETTE.items():
    subset = df[df["Diabetes_012"] == cls]["BMI"]
    axes[1].hist(subset, bins=40, alpha=0.5, color=color, label=LABEL_MAP[cls], density=True)
axes[1].set_xlabel("BMI")
axes[1].set_ylabel("Density")
axes[1].set_title("BMI Distribution by Diabetes Class")
axes[1].legend()

save(fig, "02_bmi_distribution.png")

findings.append(
    f"BMI (Chart 02): Overall median BMI = {df['BMI'].median():.0f}, mean = {df['BMI'].mean():.1f}. "
    "Diabetic patients show a rightward shift in BMI distribution (higher BMI). "
    "Pre-diabetes distribution sits between the other two classes. "
    "BMI is a strong candidate predictor for the model."
)


# CHART 03 — Age distribution
# PURPOSE  : Diabetes prevalence increases with age; visualize the age profile.
print(" 03  Age distribution ")

age_labels = {
    1:"18-24", 2:"25-29", 3:"30-34", 4:"35-39", 5:"40-44",
    6:"45-49", 7:"50-54", 8:"55-59", 9:"60-64", 10:"65-69",
    11:"70-74", 12:"75-79", 13:"80+"
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Age Distribution", fontsize=14, fontweight="bold")

# Overall count
age_counts = df["Age"].value_counts().sort_index()
axes[0].bar([age_labels[i] for i in age_counts.index], age_counts.values, color="#9C27B0", edgecolor="white")
axes[0].set_xlabel("Age Group")
axes[0].set_ylabel("Count")
axes[0].set_title("Overall Age Group Counts")
axes[0].tick_params(axis="x", rotation=45)

# Stacked bar: proportion of each class within each age group
age_class = df.groupby(["Age", "Diabetes_012"]).size().unstack(fill_value=0)
age_class_pct = age_class.div(age_class.sum(axis=1), axis=0) * 100
ax = axes[1]
bottom = np.zeros(len(age_class_pct))
for cls in [0, 1, 2]:
    ax.bar([age_labels[i] for i in age_class_pct.index],
           age_class_pct[cls].values, bottom=bottom,
           color=PALETTE[cls], label=LABEL_MAP[cls], edgecolor="white", linewidth=0.5)
    bottom += age_class_pct[cls].values
ax.set_xlabel("Age Group")
ax.set_ylabel("% within Age Group")
ax.set_title("Diabetes Class Proportion by Age Group")
ax.tick_params(axis="x", rotation=45)
ax.legend(loc="upper left")

save(fig, "03_age_distribution.png")

findings.append(
    "AGE (Chart 03): Diabetes prevalence rises sharply with age. Age groups 60-69 "
    "show the highest diabetic proportions (>25%). Younger groups (<35) have very low "
    "prevalence. Age should be among the top features for the model."
)


# CHART 04 — General Health distribution
# PURPOSE  : GenHlth (self-rated 1=Excellent … 5=Poor) is a strong subjective predictor.

print(" 04  General Health distribution ")

gh_labels = {1:"Excellent", 2:"Very Good", 3:"Good", 4:"Fair", 5:"Poor"}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("General Health (GenHlth) Distribution", fontsize=14, fontweight="bold")

gh_counts = df["GenHlth"].value_counts().sort_index()
axes[0].bar([gh_labels[i] for i in gh_counts.index], gh_counts.values,
            color="#00BCD4", edgecolor="white")
axes[0].set_ylabel("Count")
axes[0].set_title("Overall GenHlth Counts")

gh_class = df.groupby(["GenHlth", "Diabetes_012"]).size().unstack(fill_value=0)
gh_pct = gh_class.div(gh_class.sum(axis=1), axis=0) * 100
bottom = np.zeros(len(gh_pct))
for cls in [0, 1, 2]:
    axes[1].bar([gh_labels[i] for i in gh_pct.index],
                gh_pct[cls].values, bottom=bottom,
                color=PALETTE[cls], label=LABEL_MAP[cls], edgecolor="white", linewidth=0.5)
    bottom += gh_pct[cls].values
axes[1].set_ylabel("% within GenHlth Group")
axes[1].set_title("Diabetes Class Proportion by GenHlth")
axes[1].legend()

save(fig, "04_genhlth_distribution.png")

findings.append(
    "GENERAL HEALTH (Chart 04): Respondents rating their health as 'Fair' or 'Poor' "
    "have far higher diabetic proportions (>40%). Those rating 'Excellent' have <5% "
    "diabetes prevalence. GenHlth is a highly informative ordinal feature."
)


# CHART 05 — Binary feature positive rates by class
# PURPOSE  : Compare prevalence of yes/no risk factors across diabetes classes.
print(" 05  Binary feature rates by class ")

rate = df.groupby("Diabetes_012")[BINARY_COLS].mean() * 100

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(BINARY_COLS))
width = 0.25

for i, cls in enumerate([0, 1, 2]):
    ax.bar(x + i*width, rate.loc[cls], width, label=LABEL_MAP[cls],
           color=PALETTE[cls], edgecolor="white", linewidth=0.5)

ax.set_xticks(x + width)
ax.set_xticklabels(BINARY_COLS, rotation=45, ha="right")
ax.set_ylabel("% of respondents with Yes (=1)")
ax.set_title("Binary Risk Factor Rates by Diabetes Class", fontsize=14, fontweight="bold")
ax.legend()
ax.set_ylim(0, 110)

save(fig, "05_binary_feature_rates.png")

findings.append(
    "BINARY FEATURES (Chart 05): HighBP, HighChol, DiffWalk, HeartDiseaseorAttack, "
    "and Stroke all increase monotonically from No-Diabetes → Pre-Diabetes → Diabetes. "
    "PhysActivity and Veggies/Fruits show inverse trends (protective factors). "
    "HvyAlcoholConsump is actually lower in diabetics, likely due to lifestyle restrictions."
)


# CHART 06 — Correlation heatmap
# PURPOSE  : Identify multicollinearity and feature–target relationships.
print(" 06  Correlation heatmap ")

corr = df.drop(columns=["Diabetes_Label"]).corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, linewidths=0.4, ax=ax,
            annot_kws={"size": 7}, square=True)
ax.set_title("Feature Correlation Heatmap (lower triangle)", fontsize=14, fontweight="bold")

save(fig, "06_correlation_heatmap.png")

# Top correlations with target
target_corr = corr["Diabetes_012"].drop("Diabetes_012").abs().sort_values(ascending=False)
top5 = target_corr.head(5)
findings.append(
    f"CORRELATIONS (Chart 06): Top 5 features correlated with Diabetes_012: "
    + ", ".join([f"{k} ({v:.2f})" for k, v in top5.items()])
    + ". GenHlth, BMI, HighBP, DiffWalk, and HighChol are the strongest predictors. "
    "HighBP and HighChol are moderately correlated with each other (multicollinearity risk)."
)


# CHART 07 — BMI boxplot by diabetes class
# PURPOSE  : Show spread and central tendency of BMI per class clearly.
print(" 07  BMI by diabetes class ")

fig, ax = plt.subplots(figsize=(9, 6))
bp = ax.boxplot(
    [df[df["Diabetes_012"]==cls]["BMI"].values for cls in [0,1,2]],
    labels=[LABEL_MAP[cls] for cls in [0,1,2]],
    patch_artist=True,
    medianprops={"color":"black", "linewidth":2}
)
for patch, color in zip(bp["boxes"], COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel("BMI")
ax.set_title("BMI Distribution by Diabetes Class", fontsize=14, fontweight="bold")

for cls, color in PALETTE.items():
    med = df[df["Diabetes_012"]==cls]["BMI"].median()
    ax.text(cls+1, med+0.5, f"Median={med:.0f}", ha="center", fontsize=10, fontweight="bold")

save(fig, "07_bmi_by_diabetes.png")

medians = {cls: df[df["Diabetes_012"]==cls]["BMI"].median() for cls in [0,1,2]}
findings.append(
    f"BMI BY CLASS (Chart 07): Median BMI — No Diabetes: {medians[0]:.0f}, "
    f"Pre-Diabetes: {medians[1]:.0f}, Diabetes: {medians[2]:.0f}. "
    "Clear upward trend. Diabetic patients show more high-BMI outliers."
)


# CHART 08 — Age violin by diabetes class
# PURPOSE  : Richer view of age distribution shape per class.
print(" 08  Age by diabetes class ")

fig, ax = plt.subplots(figsize=(9, 6))
data_by_class = [df[df["Diabetes_012"]==cls]["Age"].values for cls in [0,1,2]]
parts = ax.violinplot(data_by_class, positions=[1,2,3], showmedians=True, showextrema=True)
for i, (pc, color) in enumerate(zip(parts["bodies"], COLORS)):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)
parts["cmedians"].set_color("black")
parts["cmedians"].set_linewidth(2)
ax.set_xticks([1,2,3])
ax.set_xticklabels([LABEL_MAP[cls] for cls in [0,1,2]])
ax.set_ylabel("Age Category (1=18-24 … 13=80+)")
ax.set_title("Age Distribution by Diabetes Class (Violin Plot)", fontsize=14, fontweight="bold")

save(fig, "08_age_by_diabetes.png")

findings.append(
    "AGE BY CLASS (Chart 08): Diabetic patients are concentrated in older age categories "
    "(median ~9 = 60-64). No-Diabetes group is younger on average (median ~6 = 45-49). "
    "The violin shapes confirm that diabetes risk accumulates with age."
)


# CHART 09 — Key risk factors grouped bar
# PURPOSE  : Focused comparison of the 6 highest-impact binary risk factors.
print(" 09  Risk factor comparison ")

RISK_FEATURES = ["HighBP", "HighChol", "HeartDiseaseorAttack", "Stroke", "DiffWalk", "PhysActivity"]
rate_risk = df.groupby("Diabetes_012")[RISK_FEATURES].mean() * 100

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(RISK_FEATURES))
width = 0.25
for i, cls in enumerate([0, 1, 2]):
    ax.bar(x + i*width, rate_risk.loc[cls], width, label=LABEL_MAP[cls],
           color=PALETTE[cls], edgecolor="white")

ax.set_xticks(x + width)
ax.set_xticklabels(RISK_FEATURES, rotation=20, ha="right")
ax.set_ylabel("% with condition/behaviour")
ax.set_title("Key Risk Factor Prevalence by Diabetes Class", fontsize=14, fontweight="bold")
ax.legend()
ax.set_ylim(0, 100)

save(fig, "09_risk_factors_by_class.png")

# CHART 10 — GenHlth and Income heatmap by diabetes class
# PURPOSE  : Explore socioeconomic dimension of diabetes prevalence.
print("10  GenHlth × Income heatmap ")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Diabetes Class Rate by General Health & Income Level",
             fontsize=13, fontweight="bold")

for ax, cls in zip(axes, [0, 1, 2]):
    pivot = df[df["Diabetes_012"] == cls].groupby(["GenHlth", "Income"]).size().unstack(fill_value=0)
    total = df.groupby(["GenHlth", "Income"]).size().unstack(fill_value=0)
    pct   = (pivot / total.replace(0, np.nan) * 100).fillna(0)
    sns.heatmap(pct, ax=ax, cmap="YlOrRd", annot=True, fmt=".0f",
                cbar_kws={"label": "% in class"},
                xticklabels=[f"I{i}" for i in range(1,9)],
                yticklabels=["Exc","VGood","Good","Fair","Poor"],
                annot_kws={"size": 7})
    ax.set_title(f"{LABEL_MAP[cls]}", fontsize=11)
    ax.set_xlabel("Income Level (1=Low … 8=High)")
    ax.set_ylabel("General Health")

save(fig, "10_genhlth_income_by_diabetes.png")

findings.append(
    "GENHLTH × INCOME (Chart 10): Diabetes rates are highest among low-income respondents "
    "with poor self-rated health. High-income + excellent health cells show near-zero "
    "diabetes prevalence. This interaction suggests socioeconomic features should be "
    "included in the model (possible interaction term)."
)


# CHART 11 — MentHlth and PhysHlth by class
# PURPOSE  : Assess mental and physical health burden linked to diabetes.

print("11  MentHlth & PhysHlth by class ")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Mental & Physical Health Days (Bad Days in Last 30) by Diabetes Class",
             fontsize=13, fontweight="bold")

for ax, col, title in zip(axes,
                          ["MentHlth", "PhysHlth"],
                          ["Mental Health (Bad Days)", "Physical Health (Bad Days)"]):
    data = [df[df["Diabetes_012"]==cls][col].values for cls in [0,1,2]]
    bp = ax.boxplot(data, labels=[LABEL_MAP[cls] for cls in [0,1,2]],
                    patch_artist=True, medianprops={"color":"black","linewidth":2})
    for patch, color in zip(bp["boxes"], COLORS):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_ylabel("Days (0-30)")
    ax.set_title(title)

save(fig, "11_menthlth_physhlth_by_diabetes.png")

findings.append(
    "MENTAL & PHYSICAL HEALTH (Chart 11): Diabetic respondents report more physically "
    "unhealthy days (median ~5 vs ~0 for no-diabetes). Mental health bad days show a "
    "smaller but still visible difference. Physical health burden is more pronounced. "
    "Both features have high zero-inflation (many people report 0 bad days)."
)


# CHART 12 — Pairplot of key continuous / ordinal features
# PURPOSE  : Check linear separability and inter-feature relationships.
print("12  Pairplot")

KEY_FEATS = ["Diabetes_012", "BMI", "Age", "GenHlth", "Income", "PhysHlth"]
sample = df[KEY_FEATS].sample(n=5000, random_state=42)

g = sns.pairplot(sample, hue="Diabetes_012", palette=PALETTE,
                 diag_kind="kde", plot_kws={"alpha": 0.3, "s": 10},
                 vars=["BMI", "Age", "GenHlth", "Income", "PhysHlth"])
g.figure.suptitle("Pairplot — Key Features (sample n=5,000)", y=1.02,
                  fontsize=13, fontweight="bold")

handles = [mpatches.Patch(color=PALETTE[k], label=LABEL_MAP[k]) for k in [0,1,2]]
g.figure.legend(handles=handles, loc="upper right", bbox_to_anchor=(1, 1))

save(g.figure, "12_pairplot_key_features.png")

findings.append(
    "PAIRPLOT (Chart 12): BMI vs GenHlth shows the clearest class separation. "
    "Diabetic class (red) clusters at higher BMI + poorer GenHlth. "
    "Pre-diabetes (orange) overlaps heavily with both other classes — this class is "
    "the hardest to separate and may require specialized handling."
)


# Summary statistics table
print("\nSummary statistics")
stats = df.drop(columns=["Diabetes_Label"]).describe().T
print(stats.to_string())




 Loading data 
  Shape: 229,772 rows × 22 columns

 01  Class distribution 
  ✓  01_class_distribution.png
 02  BMI distribution 
  ✓  02_bmi_distribution.png
 03  Age distribution 
  ✓  03_age_distribution.png
 04  General Health distribution 
  ✓  04_genhlth_distribution.png
 05  Binary feature rates by class 
  ✓  05_binary_feature_rates.png
 06  Correlation heatmap 
  ✓  06_correlation_heatmap.png
 07  BMI by diabetes class 
  ✓  07_bmi_by_diabetes.png
 08  Age by diabetes class 
  ✓  08_age_by_diabetes.png
 09  Risk factor comparison 
  ✓  09_risk_factors_by_class.png
10  GenHlth × Income heatmap 
  ✓  10_genhlth_income_by_diabetes.png
11  MentHlth & PhysHlth by class 
  ✓  11_menthlth_physhlth_by_diabetes.png
12  Pairplot
  ✓  12_pairplot_key_features.png

Summary statistics
                         count       mean       std   min   25%   50%   75%   max
Diabetes_012          229772.0   0.325640  0.724634   0.0   0.0   0.0   0.0   2.0
HighBP                229772.0   0.454459  